# 📓 01 · ETL: Loading and Cleaning the HMRC Data

## 📊 About the dataset
- **Source:** HMRC tax and NICs receipts for the UK, published on GOV.UK
- **Content:** monthly tax receipts, April 2017 to August 2026
- **Size:** 113 months and 46 tax types, in £ million (cash basis)

## ❓ Business questions
1. 💷 Which taxes bring in the most money for the UK?
2. 📅 Do receipts follow a seasonal pattern (for example, VAT peaks)?
3. 📈 How have total receipts changed since 2017?
4. 🔗 Do some taxes move together, such as Income Tax and NICs?
5. 🔮 Can we predict future monthly receipts?

## 🧭 This note book contains:
1. Load the raw data
2. Check its size and structure
3. Clean it 

In [ ]:
#cell:1
# Bring in pandas, our tool for working with tables
import pandas as pd

# Say where the file lives (go up one folder, then into data/raw)
file_path = "../data/raw/hmrc_monthly_receipts.csv"

#Read the file into a table called df (short for "data frame")
# header=5 → the real column names are on row 6, so skip the 5 intro lines
# encoding="utf-8-sig" → ignores a hidden invisible mark at the start of the file
# thousands="," → turns "53,962" into the number 53962
df = pd.read_csv(file_path, header=5, encoding="utf-8-sig", thousands=",")

# Show the size of the table: (rows, columns)
print(df.shape)

# Show the first 5 rows
df.head()

(114, 47)


,Table 1 HMRC Receipts by month,Total Paid Over,Total HMRC Receipts,Income Tax,Pay As You Earn Income Tax (included within Income Tax),Self-Assessment Income Tax (included within Income Tax),Capital Gains Tax,Apprenticeship Levy,National Insurance Contributions,PAYE NIC1 (EMP'er),...,Insurance Premium Tax,Landfill Tax,Climate Change Levy,Aggregates Levy,Soft Drinks Industry Levy,Plastic Packaging Tax,Swiss Capital Tax,Misc,Customs Duties,Penalties
0,April 2017,53962.0,56228.0,16222.0,16278.0,-106.0,3.0,[X],11721.0,7133.0,...,62.0,73.0,288.0,27.0,[X],[X],[X],[X],283.0,59.0
1,May 2017,38761.0,42703.0,12237.0,12940.0,-190.0,3.0,162,9946.0,5373.0,...,1057.0,58.0,241.0,44.0,[X],[X],[X],[X],260.0,48.0
2,June 2017,34572.0,37848.0,11941.0,12177.0,139.0,5.0,198,10147.0,5670.0,...,83.0,37.0,5.0,18.0,[X],[X],[X],[X],284.0,76.0
3,July 2017,58981.0,62167.0,20389.0,12649.0,8046.0,5.0,211,12267.0,6391.0,...,297.0,94.0,240.0,12.0,[X],[X],[X],[X],278.0,62.0
4,August 2017,41243.0,44235.0,12888.0,12171.0,1344.0,7.0,210,10549.0,6216.0,...,1123.0,68.0,181.0,47.0,[X],[X],[X],[X],300.0,60.0


**Action:** Loaded the raw HMRC file into a pandas table called `df`, skipping the intro lines at the top and using row 6 as the column names.

**Result:** 114 rows and 47 columns, starting at April 2017. Some columns contain `[X]`, so they were read as text instead of numbers.

In [ ]:
#cell:2
# Count how many columns are numbers (float64) and how many are text (object/str)
df.dtypes.value_counts()

# Show the last 3 rows (we expect to see "End of worksheet")
df.tail(3)

,Table 1 HMRC Receipts by month,Total Paid Over,Total HMRC Receipts,Income Tax,Pay As You Earn Income Tax (included within Income Tax),Self-Assessment Income Tax (included within Income Tax),Capital Gains Tax,Apprenticeship Levy,National Insurance Contributions,PAYE NIC1 (EMP'er),...,Insurance Premium Tax,Landfill Tax,Climate Change Levy,Aggregates Levy,Soft Drinks Industry Levy,Plastic Packaging Tax,Swiss Capital Tax,Misc,Customs Duties,Penalties
111,July 2026,97476.0,97999.0,40889.0,23439.0,17140.0,194.0,372,19074.0,13185.0,...,789.0,52.0,245.0,44.0,97,60,[X],[X],425.0,68.0
112,August 2026,66216.0,68886.0,24330.0,24079.0,1476.0,198.0,364,16641.0,12449.0,...,1524.0,65.0,131.0,26.0,0,4,[X],[X],454.0,100.0
113,End of worksheet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Action:** Checked what type of data each column holds, and looked at the end of the table.

**Result:** 31 columns are numbers and 16 are text. The last row says "End of worksheet", which isn't data.

In [ ]:
#cell:3
#  Rename the long first column to something short: "Month"
df = df.rename(columns={"Table 1 HMRC Receipts by month": "Month"})

#  Keep every row EXCEPT the "End of worksheet" one (it's not real data)
df = df[df["Month"] != "End of worksheet"]

#  Check the size of the table again (we expect fewer rows now)
print(df.shape)

(113, 45)


**Action:** Renamed the first column to `Month` and dropped the "End of worksheet" row.

**Result:** 113 rows by 47 columns, one row for every month from April 2017 to August 2026.

In [ ]:
#cell:4
#  Loop through every column except the first one (Month)
for col in df.columns[1:]:
    #  Remove commas, then turn text into numbers
    # errors="coerce" = anything that isn't a number (like [X]) becomes NaN (empty)
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")

**Action:** Converted every tax column into numbers. Commas were removed and every `[X]` became `NaN`, which is how pandas shows an empty value.

**Note:** `NaN` means "not available", not zero. Newer taxes only exist for part of the period, so they're mostly empty.

In [ ]:
#cell:5
#  Turn "April 2017" text into a real date (1 April 2017)
# %B = full month name, %Y = 4-digit year
df["Month"] = pd.to_datetime(df["Month"], format="%B %Y")

#  Count column types again: we expect 46 numbers + 1 date
print(df.dtypes.value_counts())

float64           46
datetime64[us]     1
Name: count, dtype: int64


**Action:** Changed the `Month` column from text into real dates, so pandas can sort and plot by time.

**Result:** 46 number columns and 1 date column, covering April 2017 to August 2026.

In [ ]:
#cell:6
# Work out what share of each column is empty
# df.isna() marks every empty cell as True, and the average of True/False gives the share
missing_pct = (df.isna().mean() * 100).round(1)

# Sort from most empty to least empty and show the top 15
missing_pct.sort_values(ascending=False).head(15)

Bank payroll tax                              100.0
Misc                                          100.0
Digital Services Tax                           95.6
Economic Crime Levy                            94.7
Swiss Capital Tax                              93.8
Electricity Generator Levy                     92.9
Diverted Profits Tax                           92.0
Residential Property Developer Tax             91.2
Energy Profits Levy                            88.5
Offshore (included within Corporation Tax)     75.2
Bank Levy                                      66.4
Bank Surcharge                                 65.5
Plastic Packaging Tax                          55.8
Soft Drinks Industry Levy                      13.3
Apprenticeship Levy                             0.9
dtype: float64

**Action:** Calculated the percentage of empty values in each column and listed the 15 emptiest.

**Result:** Bank payroll tax and Misc are empty in every month (100%). Several newer taxes are more than 88% empty, including the Digital Services Tax, Economic Crime Levy and Energy Profits Levy. All the other columns are almost complete.

**Note:** Some of these taxes, like the Digital Services Tax, only started recently, so empty doesn't mean broken. A column that is empty in every month is a different story, because it can't tell us anything.

In [ ]:
#cell:7
# Find the columns that are empty in every single month
empty_cols = missing_pct[missing_pct == 100].index.tolist()

# Print their names so we can see exactly what we're about to remove
print(empty_cols)

# Drop them, because a column with no data can't tell us anything
df = df.drop(columns=empty_cols)

# Check the new size, which should be (113, 45)
print(df.shape)

['Bank payroll tax', 'Misc']
(113, 45)


**Action:** Dropped the two columns that were empty in every month, Bank payroll tax and Misc.

**Result:** The table is now 113 rows by 45 columns.

**Note:** I kept the other mostly-empty taxes, like the Digital Services Tax, because they're real taxes that only started recently. They don't have enough months to model, so we'll leave them out of the machine learning later, but they can still appear in our exploration.


**Next notebook:** [Click here to continue to the EDA notebook](02_eda.ipynb)